# Capstone 2.1: Monthly FIDE Rating History and Player Sampling

**Purpose:**  
This notebook prepares the longitudinal FIDE Standard rating history and creates the fixed Indian youth player sample used later in EDA, feature engineering, and modelling.

**Important note:**  
This notebook documents data preparation and sampling. The final modelling results should be reproduced from the frozen modelling notebook and frozen 1,000-player sample, not by regenerating a new random sample.

**Main outputs:**  
- Cleaned monthly FIDE Standard rating history  
- Indian youth player pool  
- Player-level 12-month feature table  
- Fixed 1,000-player sample for downstream analysis


In [1]:
# ============================================================
# Step 0: Import pandas and set display options
# Purpose:
# - Load pandas for data handling.
# - Expand notebook display settings so wide FIDE tables are easier to inspect.
# ============================================================

import pandas as pd

pd.set_option("display.max_columns", None)      # show all columns
pd.set_option("display.width", 2000)            # increase print width
pd.set_option("display.max_colwidth", None)     # don't truncate long text
pd.set_option("display.max_rows", 50)           # optional: show more rows

## 1. Check downloaded monthly FIDE ZIP files

This section checks whether the downloaded monthly FIDE rating ZIP files are present and whether any files have the same size, which may indicate duplicate or failed downloads.


In [3]:
# ============================================================
# Step 2: Check downloaded ZIP file sizes
# Purpose:
# - Read all ZIP files in the raw FIDE history folder.
# - Identify duplicate file sizes that might indicate duplicate or failed downloads.
# ============================================================

from pathlib import Path
import pandas as pd

raw_dir = Path.home() / "Desktop/Arun/Arun/DBA-Walsh/SecondYr/Capstone/ML_Driven_TalentIdentification_In_Youth_Chess" / "data" / "raw"

zip_files = sorted(raw_dir.rglob("*.zip"))

size_check = []

for zip_path in zip_files:
    size_check.append({
        "zip_file": zip_path.name,
        "size_bytes": zip_path.stat().st_size,
        "size_mb": round(zip_path.stat().st_size / (1024 * 1024), 2)
    })

size_df = pd.DataFrame(size_check)

duplicate_sizes = size_df[size_df.duplicated("size_bytes", keep=False)]
duplicate_sizes = duplicate_sizes.sort_values(["size_bytes", "zip_file"])

if duplicate_sizes.empty:
    print("No ZIP files have the same exact size.And hence every month's download was unique in name and data both")
else:
    print("ZIP files with the same exact size:")
    print(duplicate_sizes.to_string(index=False))

ZIP files with the same exact size:
                 zip_file  size_bytes  size_mb
     standard_2024_01.zip    10294252     9.82
     standard_2024_01.zip    10294252     9.82
     standard_2024_02.zip    10478487     9.99
     standard_2024_02.zip    10478487     9.99
     standard_2024_03.zip    10503641    10.02
     standard_2024_03.zip    10503641    10.02
     standard_2024_04.zip    10650927    10.16
     standard_2024_04.zip    10650927    10.16
     standard_2024_05.zip    10782399    10.28
     standard_2024_05.zip    10782399    10.28
     standard_2024_07.zip    10894809    10.39
     standard_2024_07.zip    10894809    10.39
     standard_2024_06.zip    10901315    10.40
     standard_2024_06.zip    10901315    10.40
     standard_2024_08.zip    11003776    10.49
     standard_2024_08.zip    11003776    10.49
     standard_2024_09.zip    11070037    10.56
     standard_2024_09.zip    11070037    10.56
     standard_2024_10.zip    11124027    10.61
     standard_2024_10.zi

## 2. Read one FIDE Standard rating list

This section extracts one downloaded FIDE ZIP file, reads the fixed-width text file inside it, and inspects the raw columns before building the full longitudinal dataset.


In [4]:
# ============================================================
# Step 3: Extract and inspect one FIDE Standard rating list
# Purpose:
# - Unzip one FIDE rating-list file.
# - Read the fixed-width text rating file into a DataFrame.
# - Inspect columns and sample records before processing all months.
# ============================================================

# Recursively read all ZIP files from your folder

import zipfile
import pandas as pd
from pathlib import Path

# Path to downloaded ZIP file
zip_path = Path.home() / "Downloads" / "standard_rating_list.zip"

# Extract ZIP
extract_dir = Path.home() / "Downloads" / "fide_standard_rating"
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

# Find extracted TXT file
txt_files = list(extract_dir.glob("*.txt"))
print("Text file:",txt_files)

txt_path = txt_files[0]
print("**Reading:", txt_path)

# Read FIDE fixed-width TXT file
df = pd.read_fwf(txt_path)

# View first rows
print("**Few Records:\n",df.head())
print("\n**Dataframe info:\n",df.info)
      

Text file: [PosixPath('/Users/arunkumar/Downloads/fide_standard_rating/standard_rating_list.txt')]
**Reading: /Users/arunkumar/Downloads/fide_standard_rating/standard_rating_list.txt
**Few Records:
    ID Number                   Name  Fed Sex  Tit WTit OTit  FOA  APR26  Gms   K  B-day Flag
0  537001345   A Arbhin Vanniarajan  IND   M  NaN  NaN  NaN  NaN   1462    0  40   2018  NaN
1   10245154  A B M Jobair, Hossain  BAN   M  NaN  NaN  NaN  NaN   1750    0  40   1998    i
2   25121731             A C J John  IND   M  NaN  NaN  NaN  NaN   1438    0  40   1987    i
3   35077023         A Chakravarthy  IND   M  NaN  NaN  NaN  NaN   1491    0  40   1986    i
4  537070436              A Darshil  IND   M  NaN  NaN  NaN  NaN   1419    0  40   2013  NaN

**Dataframe info:
 <bound method DataFrame.info of         ID Number                   Name  Fed Sex  Tit WTit OTit  FOA  APR26  Gms   K  B-day Flag
0       537001345   A Arbhin Vanniarajan  IND   M  NaN  NaN  NaN  NaN   1462    0  40   2018 

## 3. Initial filtering: Indian and youth players

These quick filters verify that Indian federation players and youth players can be isolated from the raw monthly rating list.


In [5]:
# ============================================================
# Step 4: Filter Indian players from the raw monthly file
# Purpose:
# - Keep only records where federation is India (Fed == 'IND').
# - Quickly inspect the size and first few records of the Indian subset.
# ============================================================

#Limit to Indian Players
india_df = df[df["Fed"] == "IND"]

print("**Few Records:\n",india_df.head())
print("\n Dataframe Shape:\n",india_df.shape)

**Few Records:
    ID Number                  Name  Fed Sex  Tit WTit OTit  FOA  APR26  Gms   K  B-day Flag
0  537001345  A Arbhin Vanniarajan  IND   M  NaN  NaN  NaN  NaN   1462    0  40   2018  NaN
2   25121731            A C J John  IND   M  NaN  NaN  NaN  NaN   1438    0  40   1987    i
3   35077023        A Chakravarthy  IND   M  NaN  NaN  NaN  NaN   1491    0  40   1986    i
4  537070436             A Darshil  IND   M  NaN  NaN  NaN  NaN   1419    0  40   2013  NaN
7   46601570          A G, Perumal  IND   M  NaN  NaN  NaN  NaN   1456    0  40   1983    i

 Dataframe Shape:
 (53408, 13)


In [6]:
# ============================================================
# Step 5: Filter Indian youth players
# Purpose:
# - Keep Indian players with birth year 2008 or later.
# - This approximates the youth-player pool used in the project.
# ============================================================

#Limit to Youth Players only
youth_india_df = india_df[india_df["B-day"] >= 2008]

print("**Indian Youth Players, Few Records:\n",youth_india_df.head())
print("\n Dataframe Shape:\n",youth_india_df.shape)


**Indian Youth Players, Few Records:
     ID Number                     Name  Fed Sex  Tit WTit OTit  FOA  APR26  Gms   K  B-day Flag
0   537001345     A Arbhin Vanniarajan  IND   M  NaN  NaN  NaN  NaN   1462    0  40   2018  NaN
4   537070436                A Darshil  IND   M  NaN  NaN  NaN  NaN   1419    0  40   2013  NaN
25   33454710                A. HUDSON  IND   M  NaN  NaN  NaN  AIM   1644    0  40   2013    i
45   88105563  Aabhas Kumar Srivastava  IND   M  NaN  NaN  NaN  NaN   1946    7  40   2009  NaN
46   48765902            Aabhas Rajput  IND   M  NaN  NaN  NaN  NaN   1536    8  40   2015  NaN

 Dataframe Shape:
 (16194, 13)


In [7]:
# ============================================================
# Step 6: Inspect available source columns
# Purpose:
# - Print raw FIDE column names before feature selection and cleaning.
# ============================================================

print(df.columns)

Index(['ID Number', 'Name', 'Fed', 'Sex', 'Tit', 'WTit', 'OTit', 'FOA', 'APR26', 'Gms', 'K', 'B-day', 'Flag'], dtype='object')


## 4. Player-level spot checks

This section checks individual FIDE IDs to verify that the source file has been read correctly.


In [8]:
# ============================================================
# Step 7: Spot-check a specific player by FIDE ID
# Purpose:
# - Confirm that individual players can be located correctly in the source file.
# ============================================================

# Check column names first
print(df.columns, "\n")

# Filter by FIDE ID
player_df = df[df["ID Number"] == 48786365]

print(player_df.to_string(index=False))

#player_df = df[df["Name"] == "Ayaan Arora"]
#print(player_df.to_string(index=False))



Index(['ID Number', 'Name', 'Fed', 'Sex', 'Tit', 'WTit', 'OTit', 'FOA', 'APR26', 'Gms', 'K', 'B-day', 'Flag'], dtype='object') 

 ID Number            Name Fed Sex Tit WTit OTit FOA  APR26  Gms  K  B-day Flag
  48786365 Nova Ayer Juyal IND   M NaN  NaN  NaN NaN   1610   18 40   2016  NaN


## 5. Build full monthly Standard rating history

This section loops through all monthly FIDE ZIP files, reads each text file, identifies the monthly Standard rating column, and appends all months into one longitudinal player-month dataset.


In [9]:
# ============================================================
# Step 8: Build combined longitudinal monthly rating history
# Purpose:
# - Loop through all monthly FIDE ZIP files.
# - Extract the monthly Standard rating column from each file.
# - Append all months into a single player-month history DataFrame.
# ============================================================

import zipfile
import pandas as pd
from pathlib import Path
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

raw_dir = Path.home() / "Downloads" / "fide_history" / "raw"

zip_files = sorted(raw_dir.rglob("*.zip"))

print("ZIP files found:", len(zip_files))

all_months = []

# Month-rating column pattern: JAN24, FEB24, MAR25, APR26 etc.
month_col_pattern = re.compile(r"^(JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)\d{2}$")

for zip_path in zip_files:
    print("Reading:", zip_path.name)

    # Extract rating_month from filename like standard_2024_01.zip
    parts = zip_path.stem.split("_")
    year = parts[-2]
    month = parts[-1]
    rating_month = f"{year}-{month}"

    with zipfile.ZipFile(zip_path, "r") as z:
        txt_files = [name for name in z.namelist() if name.lower().endswith(".txt")]

        if not txt_files:
            print("No txt found in", zip_path.name)
            continue

        txt_file = txt_files[0]

        with z.open(txt_file) as f:
            temp_df = pd.read_fwf(f)

    # Clean column names
    temp_df.columns = temp_df.columns.str.strip()

    # Find the monthly rating column, e.g. JAN24, FEB24, APR26
    rating_cols = [col for col in temp_df.columns if month_col_pattern.match(str(col).strip())]

    if len(rating_cols) == 0:
        print("No rating month column found in:", zip_path.name)
        print(temp_df.columns)
        continue

    rating_col = rating_cols[0]

    # Rename monthly rating column to common name
    temp_df = temp_df.rename(columns={rating_col: "standard_rating"})

    # Add rating month
    temp_df["rating_month"] = rating_month
    temp_df["source_file"] = zip_path.name

    all_months.append(temp_df)

history_df = pd.concat(all_months, ignore_index=True)

print("\n**Combined shape:", history_df.shape)
print("\n**Few Recpords:", history_df.head(10).to_string(index=False))

ZIP files found: 28
Reading: standard_2024_01.zip
Reading: standard_2024_02.zip
Reading: standard_2024_03.zip
Reading: standard_2024_04.zip
Reading: standard_2024_05.zip
Reading: standard_2024_06.zip
Reading: standard_2024_07.zip
Reading: standard_2024_08.zip
Reading: standard_2024_09.zip
Reading: standard_2024_10.zip
Reading: standard_2024_11.zip
Reading: standard_2024_12.zip
Reading: standard_2025_01.zip
Reading: standard_2025_02.zip
Reading: standard_2025_03.zip
Reading: standard_2025_04.zip
Reading: standard_2025_05.zip
Reading: standard_2025_06.zip
Reading: standard_2025_07.zip
Reading: standard_2025_08.zip
Reading: standard_2025_09.zip
Reading: standard_2025_10.zip
Reading: standard_2025_11.zip
Reading: standard_2025_12.zip
Reading: standard_2026_01.zip
Reading: standard_2026_03.zip
Reading: standard_2026_04.zip
Reading: standard_2026_05.zip

**Combined shape: (13858699, 15)

**Few Recpords:  ID Number                     Name Fed Sex Tit WTit OTit FOA  standard_rating  Gms  K  B

## 6. Validate one player across months

This section checks one known player across months to confirm that the longitudinal rating history was constructed correctly.


In [10]:
# ============================================================
# Step 9: Validate longitudinal history for one known player
# Purpose:
# - Check that a player has expected month-by-month records after combining files.
# ============================================================

#To check one player
player_history = history_df[
    history_df["ID Number"].astype(str).str.strip() == "25977890"
].sort_values("rating_month")

print("**One sample player:537070436\n",player_history.to_string(index=False))

**One sample player:537070436
  ID Number                   Name Fed Sex Tit WTit OTit FOA  standard_rating  Gms  K  B-day Flag rating_month          source_file
  25977890          Amartya Kumar IND   M NaN  NaN  NaN NaN             1473   24 40   2011  NaN      2024-01 standard_2024_01.zip
  25977890          Amartya Kumar IND   M NaN  NaN  NaN NaN             1520    8 40   2011  NaN      2024-02 standard_2024_02.zip
  25977890          Amartya Kumar IND   M NaN  NaN  NaN NaN             1734   16 40   2011  NaN      2024-03 standard_2024_03.zip
  25977890          Amartya Kumar IND   M NaN  NaN  NaN NaN             1719    8 40   2011  NaN      2024-04 standard_2024_04.zip
  25977890          Amartya Kumar IND   M NaN  NaN  NaN NaN             1718   17 40   2011  NaN      2024-05 standard_2024_05.zip
  25977890          Amartya Kumar IND   M NaN  NaN  NaN NaN             1749   17 40   2011  NaN      2024-06 standard_2024_06.zip
  25977890          Amartya Kumar IND   M NaN  NaN  

## 7. Keep useful columns and save cleaned history

This section keeps only the columns required for downstream filtering, feature engineering, and modelling.


In [11]:
# ============================================================
# Step 10: Select useful columns for downstream analysis
# Purpose:
# - Keep identifiers, profile fields, rating, games, K-factor, birth year,
#   rating month, and source file.
# ============================================================

#To keep only useful columns
useful_cols = [
    "ID Number", "Name", "Fed", "Sex", "Tit",
    "standard_rating", "Gms", "K", "B-day",
    "rating_month", "source_file"
]

history_clean = history_df[useful_cols]

print(history_clean.head(20).to_string(index=False))

 ID Number                         Name Fed Sex Tit  standard_rating  Gms  K  B-day rating_month          source_file
  10245154        A B M Jobair, Hossain BAN   M NaN             1583    0 40   1998      2024-01 standard_2024_01.zip
  25121731                   A C J John IND   M NaN             1063    0 40   1987      2024-01 standard_2024_01.zip
  35077023               A Chakravarthy IND   M NaN             1151    0 40   1986      2024-01 standard_2024_01.zip
  10207538             A E M, Doshtagir BAN   M NaN             1840    0 40   1974      2024-01 standard_2024_01.zip
  10680810     A hamed Ashraf, Abdallah EGY   M NaN             1728    0 40   2001      2024-01 standard_2024_01.zip
   5716365              A Hamid, Harman MAS   M NaN             1325    0 40   1970      2024-01 standard_2024_01.zip
  10206612                A K M, Sourab BAN   M NaN             1598    0 20      0      2024-01 standard_2024_01.zip
   5045886                A K, Kalshyan IND   M NaN     

In [12]:
# ============================================================
# Step 11: Spot-check cleaned history for one player
# Purpose:
# - Verify that the cleaned monthly history has correct records for a sample player.
# ============================================================

history_clean
# Filter by FIDE ID
player_df = history_clean[history_clean["ID Number"] == 25977890]

print(player_df.to_string(index=False))


 ID Number                   Name Fed Sex Tit  standard_rating  Gms  K  B-day rating_month          source_file
  25977890          Amartya Kumar IND   M NaN             1473   24 40   2011      2024-01 standard_2024_01.zip
  25977890          Amartya Kumar IND   M NaN             1520    8 40   2011      2024-02 standard_2024_02.zip
  25977890          Amartya Kumar IND   M NaN             1734   16 40   2011      2024-03 standard_2024_03.zip
  25977890          Amartya Kumar IND   M NaN             1719    8 40   2011      2024-04 standard_2024_04.zip
  25977890          Amartya Kumar IND   M NaN             1718   17 40   2011      2024-05 standard_2024_05.zip
  25977890          Amartya Kumar IND   M NaN             1749   17 40   2011      2024-06 standard_2024_06.zip
  25977890          Amartya Kumar IND   M NaN             1749   17 40   2011      2024-07 standard_2024_07.zip
  25977890 Amartya Saumitra Gupta IND   M NaN             1686    7 40   2011      2024-08 standard_2024

In [13]:
# ============================================================
# Step 12: Save cleaned FIDE Standard history
# Purpose:
# - Save the combined monthly history to CSV so later notebooks can reload it.
# ============================================================

#Save all the data for retrival later
output_path = Path.home() / "Downloads" / "fide_history" / "fide_standard_history_clean.csv"

history_clean.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: /Users/arunkumar/Downloads/fide_history/fide_standard_history_clean.csv


## 8. Reload cleaned history and standardize data types

This section reloads the saved cleaned history file and converts key columns into the correct data types.


In [14]:
# ============================================================
# Step 13: Reload cleaned history file
# Purpose:
# - Reload the saved CSV to verify it can be read independently.
# - This simulates how later notebooks will consume the cleaned history.
# ============================================================

#read it directly from saved csv
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", None)

csv_path = Path.home() / "Downloads" / "fide_history" / "fide_standard_history_clean.csv"

history_df = pd.read_csv(csv_path)

print(history_df.shape)
print(history_df.head(10).to_string(index=False))

(13858699, 11)
 ID Number                     Name Fed Sex Tit  standard_rating  Gms  K  B-day rating_month          source_file
  10245154    A B M Jobair, Hossain BAN   M NaN             1583    0 40   1998      2024-01 standard_2024_01.zip
  25121731               A C J John IND   M NaN             1063    0 40   1987      2024-01 standard_2024_01.zip
  35077023           A Chakravarthy IND   M NaN             1151    0 40   1986      2024-01 standard_2024_01.zip
  10207538         A E M, Doshtagir BAN   M NaN             1840    0 40   1974      2024-01 standard_2024_01.zip
  10680810 A hamed Ashraf, Abdallah EGY   M NaN             1728    0 40   2001      2024-01 standard_2024_01.zip
   5716365          A Hamid, Harman MAS   M NaN             1325    0 40   1970      2024-01 standard_2024_01.zip
  10206612            A K M, Sourab BAN   M NaN             1598    0 20      0      2024-01 standard_2024_01.zip
   5045886            A K, Kalshyan IND   M NaN             1671    0 20 

In [15]:
# ============================================================
# Step 14: Standardize data types
# Purpose:
# - Convert FIDE ID, rating month, rating, games, K-factor, and birth year
#   into modelling-friendly formats.
# ============================================================

history_df["ID Number"] = history_df["ID Number"].astype(str).str.strip()
history_df["rating_month"] = pd.to_datetime(history_df["rating_month"], format="%Y-%m")

history_df["standard_rating"] = pd.to_numeric(history_df["standard_rating"], errors="coerce")
history_df["Gms"] = pd.to_numeric(history_df["Gms"], errors="coerce")
history_df["K"] = pd.to_numeric(history_df["K"], errors="coerce")
history_df["B-day"] = pd.to_numeric(history_df["B-day"], errors="coerce")

In [16]:
# ============================================================
# Step 15: Re-check one player after type conversion
# Purpose:
# - Confirm that player-level filtering still works after data type cleanup.
# ============================================================

player_history = history_df[
    history_df["ID Number"] == "25977890"
].sort_values("rating_month")

print(player_history.to_string(index=False))

ID Number                   Name Fed Sex Tit  standard_rating  Gms  K  B-day rating_month          source_file
 25977890          Amartya Kumar IND   M NaN             1473   24 40   2011   2024-01-01 standard_2024_01.zip
 25977890          Amartya Kumar IND   M NaN             1520    8 40   2011   2024-02-01 standard_2024_02.zip
 25977890          Amartya Kumar IND   M NaN             1734   16 40   2011   2024-03-01 standard_2024_03.zip
 25977890          Amartya Kumar IND   M NaN             1719    8 40   2011   2024-04-01 standard_2024_04.zip
 25977890          Amartya Kumar IND   M NaN             1718   17 40   2011   2024-05-01 standard_2024_05.zip
 25977890          Amartya Kumar IND   M NaN             1749   17 40   2011   2024-06-01 standard_2024_06.zip
 25977890          Amartya Kumar IND   M NaN             1749   17 40   2011   2024-07-01 standard_2024_07.zip
 25977890 Amartya Saumitra Gupta IND   M NaN             1686    7 40   2011   2024-08-01 standard_2024_08.zip
 

## 9. Create Indian youth player pool

This section filters the cleaned monthly database to Indian youth FIDE-rated players.


In [17]:
# ============================================================
# Step 16: Create Indian youth history subset
# Purpose:
# - Filter the cleaned monthly history to Indian youth players only.
# ============================================================

india_youth = history_df[
    (history_df["Fed"] == "IND") &
    (history_df["B-day"] >= 2008)
].copy()

print(india_youth.shape)
print(india_youth.head(20).to_string(index=False))

(305876, 11)
ID Number                       Name Fed Sex Tit  standard_rating  Gms  K  B-day rating_month          source_file
 88105563    Aabhas Kumar Srivastava IND   M NaN             1284   10 40   2009   2024-01-01 standard_2024_01.zip
 48706809                    Aabhash IND   M NaN             1361    0 40   2009   2024-01-01 standard_2024_01.zip
 25199080               Aadhav Ashok IND   M NaN             1005    0 40   2010   2024-01-01 standard_2024_01.zip
 25775600              Aadhesh Nambi IND   M NaN             1146    0 40   2009   2024-01-01 standard_2024_01.zip
 48726699               Aadhinya K R IND   F NaN             1097    0 40   2013   2024-01-01 standard_2024_01.zip
 33482926              Aadhityaa S R IND   M NaN             1118    0 40   2011   2024-01-01 standard_2024_01.zip
 25974068                Aadhya Jain IND   F NaN             1422    0 40   2011   2024-01-01 standard_2024_01.zip
 33476497 Aadhya Kalpeshbhai Bhavsar IND   F NaN             1095  

## 10. Sanity checks on source ZIP files

These checks confirm that each ZIP file contains a readable FIDE text file and that the rating month information is consistent.


In [18]:
# ============================================================
# Step 17: Sanity-check ZIP file contents
# Purpose:
# - Confirm each ZIP file contains a text rating-list file.
# - Record ZIP size and internal TXT file names.
# ============================================================

# Sanity Checks
import zipfile
from pathlib import Path
import pandas as pd

raw_dir = Path.home() / "Downloads" / "fide_history" / "raw"

zip_files = sorted(raw_dir.rglob("*.zip"))

zip_check = []

for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()
        txt_files = [n for n in names if n.lower().endswith(".txt")]
        
        zip_check.append({
            "zip_file": zip_path.name,
            "zip_size_mb": round(zip_path.stat().st_size / (1024 * 1024), 2),
            "txt_files_inside": txt_files
        })

zip_check_df = pd.DataFrame(zip_check)

print(zip_check_df.to_string(index=False))

            zip_file  zip_size_mb           txt_files_inside
standard_2024_01.zip         9.82    [standard_jan24frl.txt]
standard_2024_02.zip         9.99    [standard_feb24frl.txt]
standard_2024_03.zip        10.02    [standard_mar24frl.txt]
standard_2024_04.zip        10.16    [standard_apr24frl.txt]
standard_2024_05.zip        10.28    [standard_may24frl.txt]
standard_2024_06.zip        10.40    [standard_jun24frl.txt]
standard_2024_07.zip        10.39    [standard_jul24frl.txt]
standard_2024_08.zip        10.49    [standard_aug24frl.txt]
standard_2024_09.zip        10.56    [standard_sep24frl.txt]
standard_2024_10.zip        10.61    [standard_oct24frl.txt]
standard_2024_11.zip        10.71    [standard_nov24frl.txt]
standard_2024_12.zip        10.79    [standard_dec24frl.txt]
standard_2025_01.zip        10.87    [standard_jan25frl.txt]
standard_2025_02.zip        10.98    [standard_feb25frl.txt]
standard_2025_03.zip        11.06    [standard_mar25frl.txt]
standard_2025_04.zip    

In [19]:
# ============================================================
# Step 18: Sanity-check actual monthly rating column
# Purpose:
# - Inspect the rating column inside each monthly file.
# - Verify that the detected monthly rating column matches the intended period.
# ============================================================

# Sanity Checks

import re
import zipfile
import pandas as pd
from pathlib import Path

raw_dir = Path.home() / "Downloads" / "fide_history" / "raw"
zip_files = sorted(raw_dir.rglob("*.zip"))

month_col_pattern = re.compile(r"^(JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)\d{2}$")

actual_month_check = []

for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        txt_files = [n for n in z.namelist() if n.lower().endswith(".txt")]
        
        if not txt_files:
            actual_month_check.append({
                "zip_file": zip_path.name,
                "actual_rating_column": None,
                "status": "No TXT file"
            })
            continue
        
        with z.open(txt_files[0]) as f:
            temp_df = pd.read_fwf(f, nrows=5)
        
        temp_df.columns = temp_df.columns.str.strip()
        rating_cols = [c for c in temp_df.columns if month_col_pattern.match(str(c).strip())]
        
        actual_month_check.append({
            "zip_file": zip_path.name,
            "actual_rating_column": rating_cols[0] if rating_cols else None,
            "status": "OK" if rating_cols else "No month rating column found"
        })

actual_month_check_df = pd.DataFrame(actual_month_check)

print(actual_month_check_df.to_string(index=False))

            zip_file actual_rating_column status
standard_2024_01.zip                JAN24     OK
standard_2024_02.zip                FEB24     OK
standard_2024_03.zip                MAR24     OK
standard_2024_04.zip                APR24     OK
standard_2024_05.zip                MAY24     OK
standard_2024_06.zip                JUN24     OK
standard_2024_07.zip                JUL24     OK
standard_2024_08.zip                AUG24     OK
standard_2024_09.zip                SEP24     OK
standard_2024_10.zip                OCT24     OK
standard_2024_11.zip                NOV24     OK
standard_2024_12.zip                DEC24     OK
standard_2025_01.zip                JAN25     OK
standard_2025_02.zip                FEB25     OK
standard_2025_03.zip                MAR25     OK
standard_2025_04.zip                APR25     OK
standard_2025_05.zip                MAY25     OK
standard_2025_06.zip                JUN25     OK
standard_2025_07.zip                JUL25     OK
standard_2025_08.zip

## 11. Build player validation table

This section creates a unique player-level validation table from the monthly database. It is useful for checking missing birth years, federation counts, and FIDE profile URLs.


In [20]:
# ============================================================
# Step 19: Prepare player-level validation fields
# Purpose:
# - Reload cleaned history.
# - Clean key fields and create player profile URLs.
# - Build helper functions for player-level aggregation.
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

csv_path = Path.home() / "Downloads" / "fide_history" / "fide_standard_history_clean.csv"

history_df = pd.read_csv(csv_path)

history_df.columns = history_df.columns.str.strip()

history_df["ID Number"] = history_df["ID Number"].astype(str).str.strip()
history_df["rating_month"] = pd.to_datetime(history_df["rating_month"], format="%Y-%m")

history_df["standard_rating"] = pd.to_numeric(history_df["standard_rating"], errors="coerce")
history_df["Gms"] = pd.to_numeric(history_df["Gms"], errors="coerce")
history_df["K"] = pd.to_numeric(history_df["K"], errors="coerce")
history_df["B-day"] = pd.to_numeric(history_df["B-day"], errors="coerce")

# Treat 0 birth year as missing
history_df["Birth_Year_Clean"] = history_df["B-day"].replace(0, np.nan)

# Create profile URL
history_df["FIDE_Profile_URL"] = (
    "https://ratings.fide.com/profile/" + history_df["ID Number"]
)

# Function to get last non-null value
def last_valid(series):
    valid = series.dropna()
    if len(valid) == 0:
        return np.nan
    return valid.iloc[-1]

player_validation_df = (
    history_df
    .sort_values("rating_month")
    .groupby("ID Number", as_index=False)
    .agg(
        Name=("Name", "last"),
        Fed=("Fed", "last"),
        Sex=("Sex", "last"),
        Title=("Tit", "last"),
        Birth_Year=("Birth_Year_Clean", last_valid),
        Latest_Rating=("standard_rating", "last"),
        Latest_Month=("rating_month", "last"),
        Months_Available=("rating_month", "nunique"),
        FIDE_Profile_URL=("FIDE_Profile_URL", "last")
    )
)

print(player_validation_df.head(20).to_string(index=False))

ID Number                      Name Fed Sex Title  Birth_Year  Latest_Rating Latest_Month  Months_Available                          FIDE_Profile_URL
 10000011          Kabuye, Emmanuel UGA   M  None         NaN           2243   2026-05-01                28 https://ratings.fide.com/profile/10000011
 10000038            Matovu, George UGA   M  None         NaN           2225   2026-05-01                28 https://ratings.fide.com/profile/10000038
 10000046       Kamuhangire, Silver UGA   M  None         NaN           2190   2026-05-01                28 https://ratings.fide.com/profile/10000046
 10000054         Ssentongo, Edward UGA   M  None      1957.0           2250   2026-05-01                28 https://ratings.fide.com/profile/10000054
 10000062            Okoth, Joachim UGA   M  None      1950.0           2171   2026-05-01                28 https://ratings.fide.com/profile/10000062
 10000070            Anuari, Julius UGA   M  None      1982.0           1833   2026-05-01           

In [21]:
# ============================================================
# Step 20: Create a FIDE profile URL for one player
# Purpose:
# - Demonstrate how profile URLs are generated from FIDE IDs.
# ============================================================

fide_id = "25977890"

player_profile_url = f"https://ratings.fide.com/profile/{fide_id}"

print(player_profile_url)

https://ratings.fide.com/profile/25977890


### Unique player validation table from your monthly database

In [23]:
# ============================================================
# Step 21: Create unique player validation table
# Purpose:
# - Aggregate monthly records to one row per player.
# - Capture profile fields, latest rating, latest month, months available,
#   and FIDE profile URL.
# ============================================================

#create a unique player validation table from your monthly database

player_validation_df = (
    history_df
    .sort_values("rating_month")
    .groupby("ID Number", as_index=False)
    .agg(
        Name=("Name", "last"),
        Fed=("Fed", "last"),
        Sex=("Sex", "last"),
        Title=("Tit", "last"),
        Birth_Year=("B-day", "last"),
        Latest_Rating=("standard_rating", "last"),
        Latest_Month=("rating_month", "last"),
        Months_Available=("rating_month", "nunique"),
        FIDE_Profile_URL=("FIDE_Profile_URL", "last")
    )
)

print(player_validation_df.head(20).to_string(index=False))

ID Number                      Name Fed Sex Title  Birth_Year  Latest_Rating Latest_Month  Months_Available                          FIDE_Profile_URL
 10000011          Kabuye, Emmanuel UGA   M  None           0           2243   2026-05-01                28 https://ratings.fide.com/profile/10000011
 10000038            Matovu, George UGA   M  None           0           2225   2026-05-01                28 https://ratings.fide.com/profile/10000038
 10000046       Kamuhangire, Silver UGA   M  None           0           2190   2026-05-01                28 https://ratings.fide.com/profile/10000046
 10000054         Ssentongo, Edward UGA   M  None        1957           2250   2026-05-01                28 https://ratings.fide.com/profile/10000054
 10000062            Okoth, Joachim UGA   M  None        1950           2171   2026-05-01                28 https://ratings.fide.com/profile/10000062
 10000070            Anuari, Julius UGA   M  None        1982           1833   2026-05-01           

In [57]:
# ============================================================
# Step 22: Check players with missing birth year
# Purpose:
# - Identify players who cannot be used for youth filtering due to missing age data.
# ============================================================

missing_birth = player_validation_df[player_validation_df["Birth_Year"].isna()]

print(missing_birth.head(20).to_string(index=False))
print("Missing birth year count:", missing_birth.shape[0])

Empty DataFrame
Columns: [ID Number, Name, Fed, Sex, Title, Birth_Year, Latest_Rating, Latest_Month, Months_Available, FIDE_Profile_URL]
Index: []
Missing birth year count: 0


In [60]:
# ============================================================
# Step 23: Summarize missing birth year by federation
# Purpose:
# - Understand whether missing birth years are concentrated by federation.
# ============================================================

fed_counts = (
    player_validation_df[player_validation_df["Birth_Year"].isna()]
    .groupby("Fed")
    .agg(
        player_count=("ID Number", "nunique")
    )
    .reset_index()
    .sort_values("player_count", ascending=False)
)

print(fed_counts.to_string(index=False))

Empty DataFrame
Columns: [Fed, player_count]
Index: []


## 12. Count Indian youth players

This section counts the broad Indian youth player pool available from the cleaned monthly FIDE Standard rating database.


In [64]:
# ============================================================
# Step 24: Count unique Indian youth players
# Purpose:
# - Count the broad Indian youth FIDE-rated player pool.
# ============================================================

india_youth_count = player_validation_df[
    (player_validation_df["Fed"] == "IND") &
    (player_validation_df["Birth_Year"].notna()) &
    (player_validation_df["Birth_Year"] >= 2008)
]["ID Number"].nunique()

print("Unique Indian youth players:", india_youth_count)

Unique Indian youth players: 18753


#### Chess-Results will be used to extract tournament participation and performance variables, including scores, rankings, opponent strength, and event history.

In [28]:
# ============================================================
# Step 25: Optional package installation reference
# Purpose:
# - Lists packages used for web/data extraction work.
# - This line is intentionally commented so it does not run automatically.
# ============================================================

#pip install pandas requests beautifulsoup4 lxml

##### create sample of 1,000 Indian youth players


## 13. Create player-level feature dataset for sampling

This section reloads the clean history, filters Indian youth players, constructs the April 2025 to April 2026 player-level window, and creates features used for sampling.


In [84]:
# ============================================================
# Step 26: Reload cleaned history for sampling
# Purpose:
# - Load the cleaned monthly history again.
# - Clean data types and create Birth_Year for sampling logic.
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

csv_path = Path.home() / "Downloads" / "fide_history" / "fide_standard_history_clean.csv"

history_df = pd.read_csv(csv_path)

history_df.columns = history_df.columns.str.strip()

history_df["ID Number"] = history_df["ID Number"].astype(str).str.strip()
history_df["rating_month"] = pd.to_datetime(history_df["rating_month"], format="%Y-%m")

history_df["standard_rating"] = pd.to_numeric(history_df["standard_rating"], errors="coerce")
history_df["Gms"] = pd.to_numeric(history_df["Gms"], errors="coerce").fillna(0)
history_df["K"] = pd.to_numeric(history_df["K"], errors="coerce")
history_df["B-day"] = pd.to_numeric(history_df["B-day"], errors="coerce")

# Treat 0 birth year as missing
history_df["Birth_Year"] = history_df["B-day"].replace(0, np.nan)

print(history_df.shape)
print(history_df.head().to_string(index=False))

(13858699, 12)
ID Number                     Name Fed Sex Tit  standard_rating  Gms  K  B-day rating_month          source_file  Birth_Year
 10245154    A B M Jobair, Hossain BAN   M NaN             1583    0 40   1998   2024-01-01 standard_2024_01.zip      1998.0
 25121731               A C J John IND   M NaN             1063    0 40   1987   2024-01-01 standard_2024_01.zip      1987.0
 35077023           A Chakravarthy IND   M NaN             1151    0 40   1986   2024-01-01 standard_2024_01.zip      1986.0
 10207538         A E M, Doshtagir BAN   M NaN             1840    0 40   1974   2024-01-01 standard_2024_01.zip      1974.0
 10680810 A hamed Ashraf, Abdallah EGY   M NaN             1728    0 40   2001   2024-01-01 standard_2024_01.zip      2001.0


In [85]:
# ============================================================
# Step 27: Filter Indian youth player-month records
# Purpose:
# - Apply federation and birth-year criteria to create the Indian youth history.
# ============================================================

#Filter Indian youth players

#Let us define youth as U18 in 2026, meaning birth year 2008 or later.
india_youth_history = history_df[
    (history_df["Fed"] == "IND") &
    (history_df["Birth_Year"].notna()) &
    (history_df["Birth_Year"] >= 2008)
].copy()

print("Indian youth player-month records:", india_youth_history.shape)
print("Unique Indian youth players:", india_youth_history["ID Number"].nunique())

Indian youth player-month records: (305876, 12)
Unique Indian youth players: 18771


In [86]:
# ============================================================
# Step 28: Define 12-month sampling window
# Purpose:
# - Restrict records to April 2025 through April 2026.
# - This window is used to construct player-level sampling features.
# ============================================================

# Build 12-month feature dataset

# Use Apr 2025 → Apr 2026 as the 12-month prediction/evaluation window.

start_month = pd.to_datetime("2025-04")
end_month = pd.to_datetime("2026-04")

period_df = india_youth_history[
    (india_youth_history["rating_month"] >= start_month) &
    (india_youth_history["rating_month"] <= end_month)
].copy()

print(period_df["rating_month"].min(), period_df["rating_month"].max())
print(period_df.shape)
print("Unique players in period:", period_df["ID Number"].nunique())

2025-04-01 00:00:00 2026-04-01 00:00:00
(164189, 12)
Unique players in period: 18266


In [87]:
# ============================================================
# Step 29: Create player-level features for sampling
# Purpose:
# - Aggregate player-month records into one row per player.
# - Create rating gain, activity, active-month, and inactivity features.
# ============================================================

#Create player-level features

def first_valid(series):
    valid = series.dropna()
    return valid.iloc[0] if len(valid) > 0 else np.nan

def last_valid(series):
    valid = series.dropna()
    return valid.iloc[-1] if len(valid) > 0 else np.nan

player_features = (
    period_df
    .sort_values(["ID Number", "rating_month"])
    .groupby("ID Number", as_index=False)
    .agg(
        Name=("Name", "last"),
        Fed=("Fed", "last"),
        Sex=("Sex", "last"),
        Title=("Tit", "last"),
        Birth_Year=("Birth_Year", last_valid),

        rating_start=("standard_rating", first_valid),
        rating_end=("standard_rating", last_valid),

        months_available=("rating_month", "nunique"),
        total_games_12m=("Gms", "sum"),
        avg_monthly_games=("Gms", "mean"),
        max_monthly_games=("Gms", "max"),
        active_months_12m=("Gms", lambda x: (x > 0).sum()),
        inactive_months_12m=("Gms", lambda x: (x == 0).sum())
    )
)

player_features["rating_gain_12m"] = (
    player_features["rating_end"] - player_features["rating_start"]
)

player_features["age_approx_2026"] = 2026 - player_features["Birth_Year"]

# Remove players without start/end ratings
player_features = player_features[
    player_features["rating_start"].notna() &
    player_features["rating_end"].notna()
].copy()

print(player_features.shape)
print(player_features.head(20).to_string(index=False))

(18266, 16)
ID Number                     Name Fed Sex Title  Birth_Year  rating_start  rating_end  months_available  total_games_12m  avg_monthly_games  max_monthly_games  active_months_12m  inactive_months_12m  rating_gain_12m  age_approx_2026
121105400  Salunke, Shlok Prashant IND   M  None      2010.0          1638        1637                 3                5           1.666667                  5                  1                    2               -1             16.0
129105193         Tewari, Siddhant IND   M  None      2012.0          1561        1535                12                6           0.500000                  3                  2                   10              -26             14.0
 13214624         Daivik, Dayanand IND   M  None      2013.0          1466        1476                12               18           1.500000                  6                  4                    8               10             13.0
  1356615         Mogasale, Chetan IND   M  None    

## 14. Create fixed 1,000-player sample

This section creates a reproducible 1,000-player sample using a fixed random seed. The sample should be treated as fixed for final modelling so report results do not change.


In [92]:
# ============================================================
# Step 30: Create reproducible stratified 1,000-player sample
# Purpose:
# - Assign rating bands.
# - Sample players by rating band using random_state=42.
# - Adjust sample size to exactly 1,000 players.
# Important:
# - Do not change the seed or sampling logic unless final results are regenerated.
# ============================================================

# Select exactly 1,000 players
# Use a reproducible random sample. stratified sample by rating band. This is better than pure random sampling because it avoids overrepresenting only low-rated players.

player_features["rating_band"] = pd.cut(
    player_features["rating_start"],
    bins=[0, 1000, 1200, 1400, 1600, 1800, 2000, 3000],
    labels=["<1000", "1000-1199", "1200-1399", "1400-1599", "1600-1799", "1800-1999", "2000+"],
    include_lowest=True
)

print(player_features["rating_band"].value_counts(dropna=False))

player_sample_1000 = (
    player_features
    .groupby("rating_band", group_keys=False, observed=True)
    .apply(lambda x: x.sample(
        n=min(len(x), max(1, round(len(x) / len(player_features) * 1000))),
        random_state=42
    ))
    .reset_index(drop=True)
)

# If rounding gives slightly more or less than 1000, adjust
if len(player_sample_1000) > 1000:
    player_sample_1000 = player_sample_1000.sample(n=1000, random_state=42)
elif len(player_sample_1000) < 1000:
    remaining = player_features[
        ~player_features["ID Number"].isin(player_sample_1000["ID Number"])
    ]
    needed = 1000 - len(player_sample_1000)
    extra = remaining.sample(n=min(needed, len(remaining)), random_state=42)
    player_sample_1000 = pd.concat([player_sample_1000, extra], ignore_index=True)

print(player_sample_1000.shape)
print(player_sample_1000["rating_band"].value_counts(dropna=False))



rating_band
1400-1599    15761
1600-1799     1947
1800-1999      359
2000+          181
1200-1399       18
<1000            0
1000-1199        0
Name: count, dtype: int64
(1000, 17)
rating_band
1400-1599    862
1600-1799    107
1800-1999     20
2000+         10
1200-1399      1
<1000          0
1000-1199      0
Name: count, dtype: int64


/var/folders/31/z67rs8rx6cx41cfsvty5nm180000gn/T/ipykernel_68950/2674466179.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


In [94]:
# ============================================================
# Step 31: Save fixed 1,000-player sample
# Purpose:
# - Save the reproducible sample for downstream extraction, EDA, and modelling.
# ============================================================

player_sample_1000_output_path = Path.home() / "Downloads" / "fide_history" / "player_sample_1000.csv"

player_sample_1000.to_csv(player_sample_1000_output_path, index=False)


## 15. Optional balanced sample experiment

This final section creates an alternative balanced sample by rating band. It is retained for exploration only and should not replace the fixed sample used in the final report unless the report results are regenerated.


In [96]:
# ============================================================
# Step 32: Optional balanced-sample experiment
# Purpose:
# - Create an alternative sample with manually specified rating-band targets.
# - This is exploratory and should not replace the fixed final sample.
# ============================================================

# For a balanced 1,000-player sample, use:
target_per_band = {
    "1400-1599": 600,
    "1600-1799": 250,
    "1800-1999": 100,
    "2000+": 50
}

sample_parts = []

for band, n in target_per_band.items():
    band_df = player_features[player_features["rating_band"] == band]
    actual_n = min(n, len(band_df))
    
    if actual_n > 0:
        sample_parts.append(
            band_df.sample(n=actual_n, random_state=42)
        )

balanced_sample_1000 = pd.concat(sample_parts, ignore_index=True)

print(balanced_sample_1000.shape)
print(balanced_sample_1000["rating_band"].value_counts())

(1000, 17)
rating_band
1400-1599    600
1600-1799    250
1800-1999    100
2000+         50
<1000          0
1000-1199      0
1200-1399      0
Name: count, dtype: int64
